In [1]:
using Revise
Threads.nthreads()
includet("../src/ks-stat.jl")

In [ ]:
#This a serial function which estimates Pr(S > c(α))
function serial_func(m,n,alpha,r)
    c_val = calculate_c(m,n,alpha)
    count = 0
    for _ in 1:r 
        X = randn(m)
        Y = randn(n)
        ks = ks_func_2pt(X,Y)

        if ks > c_val
            count +=1
        end
    end
    return count/r
end

n=1000
m=1000
alpha = 0.05
r = 1000
result = serial_func(m,n,alpha,r)
println("Estimated Type I Error: ",result)

Estimated Type I Error: 0.05


In [63]:
#This is a parallel function that estimates Pr(S > c(α))
#修改 代码： 改为正确的Parallel 避免race condition 的形式

function parallel_func(m,n,alpha,r)
    c_val = calculate_c(m,n,alpha)
    chunk_size =cld(r,Threads.nthreads())
    chunk = Iterators.partitions(1:r,chunk_size)
    count  = 0

    tasks = map(chunk) do chunk
        Threads.@threads begin
            for _ in 1:r 
                X = randn(m)
                Y = randn(n)
                ks = ks_func_2pt(X,Y)

                if ks > c_val
                    count +=1
                end
            end
        return count
        end
    end
    
    chunks = fetch.(tasks)
    total_sum = chunks/r
    return total_sum
end


LoadError: LoadError: ArgumentError: @threads requires a `for` loop expression
in expression starting at In[63]:11

In [ ]:

@time begin
    serial_func(1000,1000,0.05,1000)
end

@time begin
    parallel_func(1000,1000,0.05,1000)
end

n=1000
m=1000
alpha = 0.05
r = 1000
result = parallel_func(m,n,alpha,r)
println("Estimated Type I Error: ",result)